In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\LENOVO\Desktop\customer-churn-prediction\data\customer_churn_cleaned.csv")

df.shape

(1000000, 31)

In [ ]:
# Splitting the data into training and testing sets

from sklearn.model_selection import train_test_split

X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(X,y,
test_size=0.20,random_state=42,stratify=y)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (800000, 30)
X_test : (200000, 30)
y_train: (800000,)
y_test : (200000,)


In [16]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000000, 30)
y shape: (1000000,)


In [ ]:
# Selecting numerical and categorical columns

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(include="str").columns.tolist()

print("Numerical columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numerical columns:
['age', 'annual_income', 'dependents', 'tenure', 'senior_citizen', 'monthlycharges', 'totalcharges', 'num_services', 'has_phone_service', 'has_internet_service', 'has_online_security', 'has_online_backup', 'has_device_protection', 'has_tech_support', 'has_streaming_tv', 'has_streaming_movies', 'customer_satisfaction', 'num_complaints', 'num_service_calls', 'late_payments', 'avg_monthly_gb', 'days_since_last_interaction', 'credit_score']

Categorical columns:
['signup_date', 'gender', 'education', 'marital_status', 'contract', 'payment_method', 'paperless_billing']


In [19]:
df["signup_date"].dtypes

<StringDtype(storage='python', na_value=nan)>

In [ ]:
# Converting signup_date to datetime

X_train["signup_date"] = pd.to_datetime(X_train["signup_date"])
X_test["signup_date"] = pd.to_datetime(X_test["signup_date"])

print(X_train["signup_date"].dtype)
print(X_test["signup_date"].dtype)

datetime64[us]
datetime64[us]


In [ ]:
#  Extracting year from signup date

X_train["signup_year"] = X_train["signup_date"].dt.year
X_test["signup_year"] = X_test["signup_date"].dt.year

print(X_train["signup_year"].value_counts().sort_index())

signup_year
2021    154465
2022    160098
2023    159994
2024    160902
2025    159776
2026      4765
Name: count, dtype: int64


In [ ]:
# Extracting month from signup date

X_train["signup_month"] = X_train["signup_date"].dt.month
X_test["signup_month"] = X_test["signup_date"].dt.month

print(X_train["signup_month"].value_counts().sort_index())

signup_month
1     67725
2     62221
3     67722
4     65192
5     68156
6     65445
7     68098
8     68100
9     66032
10    68073
11    65391
12    67845
Name: count, dtype: int64


In [24]:
# Dropping the original signup_date column

X_train = X_train.drop(columns=["signup_date"])
X_test = X_test.drop(columns=["signup_date"])

print(X_train.shape)
print(X_test.shape)

(800000, 31)
(200000, 31)


In [ ]:
# conforming categorical columns after feature engineering

categorical_cols = X_train.select_dtypes(include="str").columns.tolist()

print(categorical_cols)

['gender', 'education', 'marital_status', 'contract', 'payment_method', 'paperless_billing']


In [ ]:
# Selecting numerical and categorical columns
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()

print("Number of numerical columns:", len(numeric_cols))
print("Number of categorical columns:", len(categorical_cols))

Number of numerical columns: 25
Number of categorical columns: 6


In [27]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Creating a pipeline for numerical features

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [29]:
# Creating a pipeline for categorical features

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [30]:
# Combining the numerical and categorical pipelines into a single preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

In [31]:
# Processing the training data

X_train_processed = preprocessor.fit_transform(X_train)

In [32]:
# Processing the testing data

X_test_processed = preprocessor.transform(X_test)

In [ ]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape :", X_test_processed.shape)

X_train_processed shape: (800000, 46)
X_test_processed shape : (200000, 46)


In [ ]:
# Checking for missing values in the processed data

print("Missing values in X_train_processed:",
      np.isnan(X_train_processed).sum())

print("Missing values in X_test_processed:",
      np.isnan(X_test_processed).sum())

Missing values in X_train_processed: 0
Missing values in X_test_processed: 0


In [35]:
# Getting feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))
print(feature_names)

Number of features: 46
['num__age' 'num__annual_income' 'num__dependents' 'num__tenure'
 'num__senior_citizen' 'num__monthlycharges' 'num__totalcharges'
 'num__num_services' 'num__has_phone_service' 'num__has_internet_service'
 'num__has_online_security' 'num__has_online_backup'
 'num__has_device_protection' 'num__has_tech_support'
 'num__has_streaming_tv' 'num__has_streaming_movies'
 'num__customer_satisfaction' 'num__num_complaints'
 'num__num_service_calls' 'num__late_payments' 'num__avg_monthly_gb'
 'num__days_since_last_interaction' 'num__credit_score' 'num__signup_year'
 'num__signup_month' 'cat__gender_Female' 'cat__gender_Male'
 'cat__gender_Other' 'cat__education_bachelor' 'cat__education_college'
 'cat__education_high_school' 'cat__education_master' 'cat__education_phd'
 'cat__marital_status_divorced' 'cat__marital_status_married'
 'cat__marital_status_single' 'cat__marital_status_widowed'
 'cat__contract_month_to_month' 'cat__contract_one_year'
 'cat__contract_two_year' 'cat

In [36]:
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

y_train shape: (800000,)
y_test shape : (200000,)

Training target distribution:
churn
0    720618
1     79382
Name: count, dtype: int64


In [ ]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.joblib")

['../models/preprocessor.joblib']

Logistic Regression

In [ ]:
# Fitting the logistic regression model to the processed training data

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

In [ ]:
# Fitting the logistic regression model to the processed training data

logistic_model.fit(X_train_processed, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to u

In [ ]:
# Making predictions on the processed testing data

y_pred = logistic_model.predict(X_test_processed)

print(y_pred[:20])

[0 0 1 1 1 0 1 1 0 1 0 1 0 1 0 0 0 1 1 0]


In [41]:
# Evaluating the model's performance using a confusion matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[112849  67306]
 [  7155  12690]]


In [42]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.627695
Precision: 0.15863293164658232
Recall   : 0.6394557823129252
F1 Score : 0.25420418465359923


In [43]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.63      0.75    180155
           1       0.16      0.64      0.25     19845

    accuracy                           0.63    200000
   macro avg       0.55      0.63      0.50    200000
weighted avg       0.86      0.63      0.70    200000



Random Forest 

In [ ]:
# Fitting the random forest model to the processed training data

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [45]:
# Fitting the random forest model to the processed training data

rf_model.fit(X_train_processed, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [46]:
y_pred_rf = rf_model.predict(X_test_processed)

print(y_pred_rf[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [47]:
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

[[179146   1009]
 [ 19279    566]]


In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)

Accuracy : 0.89856
Precision: 0.3593650793650794
Recall   : 0.02852103804484757
F1 Score : 0.05284780578898226


xgboost

In [49]:
from xgboost import XGBClassifier

In [50]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=9.08,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

In [51]:
xgb_model.fit(X_train_processed, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [52]:
y_pred_xgb = xgb_model.predict(X_test_processed)

print(y_pred_xgb[:20])

[0 0 1 1 0 0 1 0 0 1 0 1 0 1 0 0 0 1 1 0]


In [53]:
from sklearn.metrics import confusion_matrix

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print(cm_xgb)

[[113930  66225]
 [  7303  12542]]


In [54]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print("Accuracy :", accuracy_xgb)
print("Precision:", precision_xgb)
print("Recall   :", recall_xgb)
print("F1 Score :", f1_xgb)

Accuracy : 0.63236
Precision: 0.15922911879340332
Recall   : 0.6319979843789367
F1 Score : 0.25437066482781


Create StratifiedKFold

In [55]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [61]:
from sklearn.model_selection import cross_validate

cv_results_lr = cross_validate(
    logistic_model,
    X_train_processed,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

In [62]:
print("Logistic Regression CV Results")
print("Mean Accuracy :", cv_results_lr["test_accuracy"].mean())
print("Mean Precision:", cv_results_lr["test_precision"].mean())
print("Mean Recall   :", cv_results_lr["test_recall"].mean())
print("Mean F1       :", cv_results_lr["test_f1"].mean())

Logistic Regression CV Results
Mean Accuracy : 0.62836875
Mean Precision: 0.15905815320381372
Mean Recall   : 0.6403340943962748
Mean F1       : 0.2548183138498067


In [56]:
# cross-validation Random Forest

from sklearn.model_selection import cross_validate


cv_results_rf = cross_validate(
    rf_model,
    X_train_processed,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

In [57]:
print("Random Forest CV Results")
print("Mean Accuracy :", cv_results_rf["test_accuracy"].mean())
print("Mean Precision:", cv_results_rf["test_precision"].mean())
print("Mean Recall   :", cv_results_rf["test_recall"].mean())
print("Mean F1       :", cv_results_rf["test_f1"].mean())

Random Forest CV Results
Mean Accuracy : 0.89875875
Mean Precision: 0.36397593475501183
Mean Recall   : 0.027147198751526064
Mean F1       : 0.050523447126657725


In [58]:
# cross-validation XGBoost

cv_results_xgb = cross_validate(
    xgb_model,
    X_train_processed,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

In [59]:
print("XGBoost CV Results")
print("Mean Accuracy :", cv_results_xgb["test_accuracy"].mean())
print("Mean Precision:", cv_results_xgb["test_precision"].mean())
print("Mean Recall   :", cv_results_xgb["test_recall"].mean())
print("Mean F1       :", cv_results_xgb["test_f1"].mean())

XGBoost CV Results
Mean Accuracy : 0.6350337500000001
Mean Precision: 0.16024811035577974
Mean Recall   : 0.6315285577605735
Mean F1       : 0.2556280386568348


Compare the models

In [64]:

cv_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        cv_results_lr["test_accuracy"].mean(),
        cv_results_rf["test_accuracy"].mean(),
        cv_results_xgb["test_accuracy"].mean()
    ],
    "Precision": [
        cv_results_lr["test_precision"].mean(),
        cv_results_rf["test_precision"].mean(),
        cv_results_xgb["test_precision"].mean()
    ],
    "Recall": [
        cv_results_lr["test_recall"].mean(),
        cv_results_rf["test_recall"].mean(),
        cv_results_xgb["test_recall"].mean()
    ],
    "F1": [
        cv_results_lr["test_f1"].mean(),
        cv_results_rf["test_f1"].mean(),
        cv_results_xgb["test_f1"].mean()
    ]
})

cv_comparison

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.628369,0.159058,0.640334,0.254818
1,Random Forest,0.898759,0.363976,0.027147,0.050523
2,XGBoost,0.635034,0.160248,0.631529,0.255628


Hyperparameter tuning 

In [65]:
from sklearn.model_selection import RandomizedSearchCV

xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

In [66]:
xgb_random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_grid,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

In [67]:
xgb_random_search.fit(X_train_processed, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'min_child_weight': [1, 3, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` c

In [68]:
print("Best Parameters:")
print(xgb_random_search.best_params_)

print("\nBest CV F1 Score:")
print(xgb_random_search.best_score_)

Best Parameters:
{'subsample': 0.7, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 1.0}

Best CV F1 Score:
0.2557525379959901


In [69]:
best_xgb_model = xgb_random_search.best_estimator_

In [70]:
y_pred_xgb_tuned = best_xgb_model.predict(X_test_processed)

In [71]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Tuned XGBoost Test Results")
print("Accuracy :", accuracy_score(y_test, y_pred_xgb_tuned))
print("Precision:", precision_score(y_test, y_pred_xgb_tuned))
print("Recall   :", recall_score(y_test, y_pred_xgb_tuned))
print("F1       :", f1_score(y_test, y_pred_xgb_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_tuned))

Tuned XGBoost Test Results
Accuracy : 0.63061
Precision: 0.1588782686650084
Recall   : 0.6340639959687578
F1       : 0.2540890917168127

Confusion Matrix:
[[113539  66616]
 [  7262  12583]]


In [72]:
best_xgb_model.predict(X_test_processed)

array([0, 0, 1, ..., 0, 0, 0], shape=(200000,))

In [73]:
X_train_model, X_val, y_train_model, y_val = train_test_split(
    X_train_processed,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [74]:
best_xgb_model.fit(X_train_model, y_train_model)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1.0
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [75]:
y_val_proba = best_xgb_model.predict_proba(X_val)[:, 1]

In [76]:
y_val_proba[:10]

array([0.25071153, 0.18207167, 0.45199615, 0.35828987, 0.7313175 ,
       0.6326828 , 0.5539369 , 0.3662707 , 0.5826333 , 0.3428486 ],
      dtype=float32)

In [ ]:
# Compare thresholds

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1
0,0.1,0.099232,1.000000,0.180549
1,0.2,0.101504,0.992504,0.184173
2,0.3,0.114034,0.932099,0.203208
3,0.4,0.132233,0.820295,0.227752
4,0.5,0.161234,0.632779,0.256987
5,0.6,0.203205,0.377047,0.264084
6,0.7,0.276543,0.155203,0.198822
7,0.8,0.411672,0.032880,0.060896
8,0.9,0.666667,0.000252,0.000504


In [ ]:
# Find the best threshold more precisely

thresholds_fine = np.arange(0.40, 0.71, 0.01)

fine_results = []

for threshold in thresholds_fine:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    fine_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred)
    })

fine_threshold_df = pd.DataFrame(fine_results)

fine_threshold_df.loc[
    fine_threshold_df["F1"].idxmax()
]

Threshold    0.570000
Precision    0.189331
Recall       0.457168
F1           0.267769
Name: 17, dtype: float64

In [79]:
y_test_proba = best_xgb_model.predict_proba(X_test_processed)[:, 1]

In [80]:
best_threshold = 0.57

y_test_pred_tuned = (y_test_proba >= best_threshold).astype(int)

In [81]:
print("Final XGBoost Test Results")
print("Threshold :", best_threshold)
print("Accuracy  :", accuracy_score(y_test, y_test_pred_tuned))
print("Precision :", precision_score(y_test, y_test_pred_tuned))
print("Recall    :", recall_score(y_test, y_test_pred_tuned))
print("F1        :", f1_score(y_test, y_test_pred_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_tuned))

Final XGBoost Test Results
Threshold : 0.57
Accuracy  : 0.75217
Precision : 0.18996056831695562
Recall    : 0.45880574452003026
F1        : 0.26867917847025496

Confusion Matrix:
[[141329  38826]
 [ 10740   9105]]


In [82]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost Baseline",
        "XGBoost Tuned",
        "XGBoost Tuned + Threshold 0.57"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_xgb_tuned),
        accuracy_score(y_test, y_test_pred_tuned)
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb),
        precision_score(y_test, y_pred_xgb_tuned),
        precision_score(y_test, y_test_pred_tuned)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb),
        recall_score(y_test, y_pred_xgb_tuned),
        recall_score(y_test, y_test_pred_tuned)
    ],
    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb),
        f1_score(y_test, y_pred_xgb_tuned),
        f1_score(y_test, y_test_pred_tuned)
    ]
})

final_model_comparison

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.627695,0.158633,0.639456,0.254204
1,Random Forest,0.898560,0.359365,0.028521,0.052848
2,XGBoost Baseline,0.632360,0.159229,0.631998,0.254371
3,XGBoost Tuned,0.630610,0.158878,0.634064,0.254089
4,XGBoost Tuned + Threshold 0.57,0.752170,0.189961,0.458806,0.268679


In [83]:
import joblib

joblib.dump(best_xgb_model, "../models/xgb_churn_model.joblib")

['../models/xgb_churn_model.joblib']

In [84]:
joblib.dump(best_threshold, "../models/churn_threshold.joblib")

['../models/churn_threshold.joblib']

In [85]:
import joblib

loaded_preprocessor = joblib.load("../models/preprocessor.joblib")
loaded_model = joblib.load("../models/xgb_churn_model.joblib")
loaded_threshold = joblib.load("../models/churn_threshold.joblib")

print("Preprocessor loaded:", type(loaded_preprocessor))
print("Model loaded:", type(loaded_model))
print("Threshold:", loaded_threshold)

Preprocessor loaded: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Model loaded: <class 'xgboost.sklearn.XGBClassifier'>
Threshold: 0.57


In [86]:
best_xgb_model.fit(
    X_train_processed,
    y_train
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,1.0
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [87]:
joblib.dump(
    best_xgb_model,
    "../models/xgb_churn_model.joblib"
)

['../models/xgb_churn_model.joblib']

Final sanity check

In [88]:
y_test_proba_final = best_xgb_model.predict_proba(X_test_processed)[:, 1]

y_test_pred_final = (
    y_test_proba_final >= best_threshold
).astype(int)

In [89]:
print("Final Model Test Results")
print("Threshold :", best_threshold)
print("Accuracy  :", accuracy_score(y_test, y_test_pred_final))
print("Precision :", precision_score(y_test, y_test_pred_final))
print("Recall    :", recall_score(y_test, y_test_pred_final))
print("F1        :", f1_score(y_test, y_test_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_final))

Final Model Test Results
Threshold : 0.57
Accuracy  : 0.750475
Precision : 0.18970642883686362
Recall    : 0.46303854875283446
F1        : 0.26914458942928693

Confusion Matrix:
[[140906  39249]
 [ 10656   9189]]
